# 04b - Poligonizar simple

Pipeline limpio:

1. Lee raster de probabilidad generado por el notebook 03.
2. Usa el threshold guardado en el checkpoint del modelo.
3. Poligoniza pixeles con probabilidad >= threshold.
4. Elimina poligonos menores a 500 m2.
5. Rellena agujeros internos menores a 300 m2.
6. Suaviza bordes, simplifica vertices y evita vaciar el resultado si el filtro de borde elimina todo.

No detecta caminos, no esqueletiza y no aplica heuristicas internas de corredores.


In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from IPython.display import display
from rasterio.windows import Window

from src.rodal_config import VALIDATION_DIR, MODEL_DIR, PRED_DIR, SHAPE_DIR, OUTPUTS_DIR, DEFAULT_PIXEL_RES_M, ensure_dirs
from src.rodal_model import load_checkpoint, checkpoint_stats
from src.rodal_bands import band_map
import src.rodal_vectorize as rodal_vectorize
importlib.reload(rodal_vectorize)
from src.rodal_vectorize import VectorizeParams, probability_to_rodal_mask, mask_to_polygons

ensure_dirs()


In [ ]:
# Configuracion
MODEL_RUN_NAME = 'v128_final_aug2026_jaccard'
PRED_RUN_NAME = 'v128_final_aug2026_jaccard_stride32'

MODEL_RUN_DIR = MODEL_DIR.parent / f'02_modelo_{MODEL_RUN_NAME}'
PRED_SOURCE_DIR = PRED_DIR / PRED_RUN_NAME

# Threshold desde checkpoint. Para forzar manualmente, poner un numero, por ejemplo 0.45.
THRESHOLD_OVERRIDE = None

MIN_POLYGON_AREA_M2 = 500
MIN_HOLE_AREA_M2 = 300
PIXEL_RES_M = DEFAULT_PIXEL_RES_M
BAND_ORDER = 'BGRNIR'
SMOOTH_ITERATIONS = 10
SIMPLIFY_TOL_M = 0.3
SMOOTH_BUFFER_M = 0.0
REMOVE_EDGE_POLYGONS = False
EDGE_BUFFER_M = 0.0
PRESERVE_HOLE_ELONGATION = True
HOLE_MIN_ELONGATION = 4.0
HOLE_MIN_LENGTH_M = 30.0

POSTPROCESS_RUN_NAME = 'poligonos_simple_thr_modelo_min500_holes300_noedge_smooth10_simplify03'
SHAPE_OUT_DIR = SHAPE_DIR.parent / f'04b_shapefiles_{PRED_RUN_NAME}_{POSTPROCESS_RUN_NAME}'
SHAPE_OUT_DIR.mkdir(parents=True, exist_ok=True)
FORCE_OVERWRITE_OUTPUTS = True

ckpt, _, _ = load_checkpoint(MODEL_RUN_DIR / 'unet_rodalizacion.pth', device='cpu')
_, _, CKPT_THRESHOLD = checkpoint_stats(ckpt)
THRESHOLD = float(CKPT_THRESHOLD if THRESHOLD_OVERRIDE is None else THRESHOLD_OVERRIDE)

vparams = VectorizeParams(
    threshold=THRESHOLD,
    close_radius_px=0,
    fill_holes=False,
    min_area_m2=MIN_POLYGON_AREA_M2,
    min_hole_area_m2=MIN_HOLE_AREA_M2,
    preserve_hole_elongation=PRESERVE_HOLE_ELONGATION,
    hole_min_elongation=HOLE_MIN_ELONGATION,
    hole_min_length_m=HOLE_MIN_LENGTH_M,
    simplify_tol_m=SIMPLIFY_TOL_M,
    smooth_buffer_m=SMOOTH_BUFFER_M,
    smooth_iterations=SMOOTH_ITERATIONS,
    remove_edge_polygons=REMOVE_EDGE_POLYGONS,
    edge_buffer_m=EDGE_BUFFER_M,
    pixel_res_m=PIXEL_RES_M,
)

print(f'Modelo: {MODEL_RUN_DIR}')
print(f'Predicciones: {PRED_SOURCE_DIR}')
print(f'Threshold checkpoint={CKPT_THRESHOLD:.3f} | usado={THRESHOLD:.3f}')
print(f'Min poligono={MIN_POLYGON_AREA_M2} m2 | min agujero={MIN_HOLE_AREA_M2} m2')
print(f'Smooth={SMOOTH_ITERATIONS} | simplify={SIMPLIFY_TOL_M}m | remove_edge={REMOVE_EDGE_POLYGONS} edge_buffer={EDGE_BUFFER_M}m')
print(f'Preservar agujeros elongados={PRESERVE_HOLE_ELONGATION} elong>={HOLE_MIN_ELONGATION} length>={HOLE_MIN_LENGTH_M}m')
print(f'Salida: {SHAPE_OUT_DIR}')


In [ ]:
prob_files = sorted(PRED_SOURCE_DIR.glob('*_prob.tif'))
if not prob_files:
    raise FileNotFoundError(f'No encontre *_prob.tif en {PRED_SOURCE_DIR}. Corre primero el notebook 03.')

rows = []
for i, path in enumerate(prob_files):
    with rasterio.open(path) as src:
        rows.append({
            'idx': i,
            'archivo': path.name,
            'width': src.width,
            'height': src.height,
            'crs': str(src.crs),
        })
prob_df = pd.DataFrame(rows)
display(prob_df)

# Dejar vacio para procesar todos. Ejemplo: SELECTED_IDXS = [0, 2]
SELECTED_IDXS = []
if SELECTED_IDXS:
    prob_files = [prob_files[i] for i in SELECTED_IDXS]
print('A procesar:', [p.name for p in prob_files])


In [ ]:
# Barrido simple de area por threshold para entender sensibilidad.
thresholds = sorted(set([0.30, 0.40, 0.50, 0.60, 0.70, round(THRESHOLD, 2)]))
rows = []
for prob_path in prob_files:
    with rasterio.open(prob_path) as src:
        prob = src.read(1).astype(np.float32)
        pixel_area = abs(src.transform.a * src.transform.e)
    for thr in thresholds:
        mask = prob >= float(thr)
        rows.append({
            'imagen': prob_path.stem.replace('_prob', ''),
            'threshold': round(float(thr), 2),
            'area_ha_raw': round(mask.sum() * pixel_area / 10000, 1),
        })
area_df = pd.DataFrame(rows)
display(area_df.pivot(index='imagen', columns='threshold', values='area_ha_raw'))


In [ ]:
def matching_image(prob_path):
    name = prob_path.stem.replace('_prob', '')
    candidates = sorted(VALIDATION_DIR.glob(f'{name}*.tif'))
    return candidates[0] if candidates else None

def vectorize_with_edge_fallback(mask, transform, crs, params):
    gdf = mask_to_polygons(mask, transform, crs, params)
    used_edge_filter = bool(params.remove_edge_polygons)
    if len(gdf) == 0 and params.remove_edge_polygons and mask.any():
        fallback_params = VectorizeParams(
            threshold=params.threshold,
            close_radius_px=params.close_radius_px,
            fill_holes=params.fill_holes,
            min_area_m2=params.min_area_m2,
            min_hole_area_m2=params.min_hole_area_m2,
            preserve_hole_elongation=params.preserve_hole_elongation,
            hole_min_elongation=params.hole_min_elongation,
            hole_min_length_m=params.hole_min_length_m,
            simplify_tol_m=params.simplify_tol_m,
            smooth_buffer_m=params.smooth_buffer_m,
            smooth_iterations=params.smooth_iterations,
            smooth_max_points=params.smooth_max_points,
            remove_edge_polygons=False,
            edge_buffer_m=params.edge_buffer_m,
            pixel_res_m=params.pixel_res_m,
        )
        gdf = mask_to_polygons(mask, transform, crs, fallback_params)
        used_edge_filter = False
    return gdf, used_edge_filter

results = []
for prob_path in prob_files:
    name = prob_path.stem.replace('_prob', '')
    print()
    print(f'=== {name} ===')
    with rasterio.open(prob_path) as src:
        prob = src.read(1).astype(np.float32)
        transform = src.transform
        crs = src.crs
        pixel_area = abs(src.transform.a * src.transform.e)

    mask = probability_to_rodal_mask(prob, vparams)
    raw_area_ha = float((prob >= THRESHOLD).sum() * pixel_area / 10000)
    mask_area_ha = float(mask.sum() * pixel_area / 10000)
    gdf, used_edge_filter = vectorize_with_edge_fallback(mask, transform, crs, vparams)
    if not used_edge_filter and REMOVE_EDGE_POLYGONS and mask.any():
        print('  Aviso: el filtro de borde eliminaba todos los poligonos; se guardo sin remove_edge_polygons para este raster.')

    out_gpkg = SHAPE_OUT_DIR / f'{name}_poligonos_unet.gpkg'
    if FORCE_OVERWRITE_OUTPUTS and out_gpkg.exists():
        out_gpkg.unlink()
    gdf.to_file(out_gpkg, driver='GPKG')

    final_area_ha = float(gdf.area_m2.sum() / 10000) if len(gdf) else 0.0
    print(f'  raw_area={raw_area_ha:.1f} ha | mask_area={mask_area_ha:.1f} ha | poligonos={len(gdf)} | final_area={final_area_ha:.1f} ha')
    print(f'  guardado: {out_gpkg.name}')
    results.append({
        'name': name,
        'prob_path': prob_path,
        'tif_path': matching_image(prob_path),
        'mask': mask,
        'gdf': gdf,
        'raw_area_ha': raw_area_ha,
        'mask_area_ha': mask_area_ha,
        'final_area_ha': final_area_ha,
        'used_edge_filter': used_edge_filter,
        'out_gpkg': out_gpkg,
    })


In [ ]:
summary = []
for r in results:
    gdf = r['gdf']
    summary.append({
        'imagen': r['name'],
        'poligonos': len(gdf),
        'area_raw_ha': round(r['raw_area_ha'], 1),
        'area_mask_ha': round(r['mask_area_ha'], 1),
        'area_final_ha': round(r['final_area_ha'], 1),
        'area_media_m2': round(gdf.area_m2.mean(), 1) if len(gdf) else 0,
    })
display(pd.DataFrame(summary))
print(f'Archivos en: {SHAPE_OUT_DIR}')
for path in sorted(SHAPE_OUT_DIR.glob('*.gpkg')):
    print(' ', path.name)


In [ ]:
# QA visual sencillo: probabilidad, mascara y poligonos finales.
P = 1800
bm = band_map(BAND_ORDER)
for r in results:
    name = r['name']
    prob_path = r['prob_path']
    tif_path = r['tif_path']
    mask = r['mask']
    gdf = r['gdf']

    with rasterio.open(prob_path) as src:
        prob = src.read(1)
        h, w = prob.shape
        transform = src.transform
    y0 = max(h // 2 - P // 2, 0); x0 = max(w // 2 - P // 2, 0)
    y1 = min(y0 + P, h); x1 = min(x0 + P, w)

    if tif_path is not None:
        with rasterio.open(tif_path) as src:
            rgb = src.read([bm.red, bm.green, bm.blue], window=Window(x0, y0, x1-x0, y1-y0)).transpose(1, 2, 0).astype(np.float32)
        valid = rgb[:, :, 0] > 0
        p2, p98 = np.percentile(rgb[valid], [2, 98]) if valid.any() else (0, 1)
        rgb = np.clip((rgb - p2) / max(p98 - p2, 1), 0, 1)
    else:
        rgb = np.zeros((y1-y0, x1-x0, 3), dtype=np.float32)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(prob[y0:y1, x0:x1], cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'Probabilidad thr={THRESHOLD:.2f}')
    axes[0].axis('off')

    axes[1].imshow(mask[y0:y1, x0:x1], cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Mascara threshold + area/huecos')
    axes[1].axis('off')

    axes[2].imshow(rgb)
    left, top = rasterio.transform.xy(transform, y0, x0, offset='ul')
    right, bottom = rasterio.transform.xy(transform, y1, x1, offset='ul')
    crop = gdf.cx[left:right, bottom:top]
    if len(crop):
        crop.plot(ax=axes[2], facecolor='none', edgecolor='lime', linewidth=1.0)
    axes[2].set_title(f'Poligonos finales: {len(gdf)}')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / f'unet_04b_{name}_poligonos_simple_qa.png', dpi=130, bbox_inches='tight')
    plt.show()
